# Blood Atlas Stage 0 — heavy local subsampling

This one-off notebook reduces the full contributor-provided Blood Atlas
(~1.9M cells) to the manageable checkpoint used by the public workflow.

**Important expression provenance**

Inspection of the downloaded `all_pbmcs` AnnData established that:

- `adata.X` is floating-point normalized/transformed expression;
- `adata.raw` is absent;
- no genuine gene-level raw-count layer is present.

Therefore this notebook intentionally:

- preserves `adata.X` exactly as supplied;
- does **not** create `layers["counts"]`;
- does **not** normalize or log-transform expression;
- does **not** run scVI.

The public/standard workflow begins from the resulting:

`data/blood_atlas/blood_atlas_subsampled.h5ad`

In [1]:
from pathlib import Path
import json

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

# ------------------------------------------------------------------
# EDIT THESE EXTERNAL PATHS IF NEEDED.
# ------------------------------------------------------------------
RAW_DIR = Path("/fastscratch/myscratch/xchen5/all_pbmcs")
ADATA_PATH = RAW_DIR / "all_pbmcs_rna.h5ad"
META_PATH = RAW_DIR / "all_pbmcs_metadata.csv"

REPO_ROOT = Path(
    "~/scRNA-cross-donor-generalization"
).expanduser()

OUT_DIR = REPO_ROOT / "data" / "blood_atlas"
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_H5AD = (
    OUT_DIR
    / "blood_atlas_subsampled.h5ad"
)

QC_DIR = (
    REPO_ROOT
    / "results"
    / "preprocessing"
    / "blood_atlas_stage0"
)
QC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Expression:", ADATA_PATH)
print("Metadata:  ", META_PATH)
print("Output:    ", OUT_H5AD)

Expression: /fastscratch/myscratch/xchen5/all_pbmcs/all_pbmcs_rna.h5ad
Metadata:   /fastscratch/myscratch/xchen5/all_pbmcs/all_pbmcs_metadata.csv
Output:     /users/xchen5/scRNA-cross-donor-generalization/data/blood_atlas/blood_atlas_subsampled.h5ad


## Load the full external source

In [2]:
if not ADATA_PATH.exists():
    raise FileNotFoundError(
        ADATA_PATH
    )

if not META_PATH.exists():
    raise FileNotFoundError(
        META_PATH
    )

adata = ad.read_h5ad(
    ADATA_PATH
)

meta = pd.read_csv(
    META_PATH
)

print(adata)
print(
    "metadata shape:",
    meta.shape,
)

assert (
    adata.n_obs
    == meta.shape[0]
), (
    f"Metadata rows "
    f"({meta.shape[0]:,}) do not "
    f"match AnnData cells "
    f"({adata.n_obs:,})."
)

AnnData object with n_obs × n_vars = 1916367 × 36601
    layers: None (.X)
metadata shape: (1916367, 20)


## Confirm and record expression provenance

In [3]:
def inspect_matrix(
    name,
    X,
    n=100_000,
):
    if X is None:
        return {
            "name": name,
            "present": False,
        }

    vals = (
        X.data
        if sparse.issparse(X)
        else np.asarray(X).ravel()
    )

    vals = vals[
        : min(n, len(vals))
    ]

    return {
        "name": name,
        "present": True,
        "dtype": str(X.dtype),
        "min": (
            float(vals.min())
            if len(vals)
            else None
        ),
        "max": (
            float(vals.max())
            if len(vals)
            else None
        ),
        "integer_like": bool(
            np.allclose(
                vals,
                np.round(vals),
                atol=1e-6,
            )
        ),
    }

diagnostics = [
    inspect_matrix(
        "adata.X",
        adata.X,
    )
]

if adata.raw is not None:
    diagnostics.append(
        inspect_matrix(
            "adata.raw.X",
            adata.raw.X,
        )
    )

for layer in adata.layers.keys():
    diagnostics.append(
        inspect_matrix(
            f"layer:{layer}",
            adata.layers[layer],
        )
    )

display(
    pd.DataFrame(
        diagnostics
    )
)

x_diag = diagnostics[0]

assert (
    x_diag["integer_like"]
    is False
), (
    "Expected the contributor-provided "
    "Blood Atlas X matrix to be transformed "
    "expression, but it appears integer-like. "
    "Re-check source provenance before proceeding."
)

assert adata.raw is None, (
    "Unexpected adata.raw detected. "
    "Inspect it before proceeding."
)

# Do NOT manufacture a raw-count layer.
if "counts" in adata.layers:
    raise ValueError(
        "An unexpected 'counts' layer is present. "
        "Inspect its provenance before proceeding."
    )

print(
    "Blood Atlas source: normalized/transformed "
    "expression only."
)
print(
    "Preserving adata.X exactly as supplied."
)

,name,present,dtype,min,max,integer_like
0,adata.X,True,float32,0.943851,6.821375,False
1,layer:None,True,float32,0.943851,6.821375,False


Blood Atlas source: normalized/transformed expression only.
Preserving adata.X exactly as supplied.


## Attach and standardize metadata

In [4]:
meta = meta.copy()
meta.index = adata.obs_names
adata.obs = meta

adata.obs["donor_id"] = (
    adata.obs["Donor_id"]
    .astype(str)
)
adata.obs["cell_type"] = (
    adata.obs["Cluster_names"]
    .astype(str)
)
adata.obs["batch"] = (
    adata.obs["Batch"]
    .astype(str)
)
adata.obs["sample_id"] = (
    adata.obs["File_name"]
    .astype(str)
)

adata.obs["age"] = (
    pd.to_numeric(
        adata.obs["Age"],
        errors="coerce",
    )
)
adata.obs["age_group"] = (
    adata.obs["Age_group"]
    .astype(str)
)
adata.obs["sex"] = (
    adata.obs["Sex"]
    .astype(str)
)

required = [
    "donor_id",
    "cell_type",
    "batch",
]

mask = np.ones(
    adata.n_obs,
    dtype=bool,
)

for col in required:
    raw = adata.obs[col]
    vals = raw.astype(str)

    mask &= raw.notna().to_numpy()
    mask &= ~vals.isin(
        [
            "nan",
            "None",
            "NA",
            "",
        ]
    ).to_numpy()

adata = adata[
    mask
].copy()

print(adata)
print(
    "donors:",
    adata.obs[
        "donor_id"
    ].nunique(),
)
print(
    "cell types:",
    adata.obs[
        "cell_type"
    ].nunique(),
)
print(
    "batches:",
    adata.obs[
        "batch"
    ].nunique(),
)

AnnData object with n_obs × n_vars = 1916367 × 36601
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers', 'donor_id', 'cell_type', 'batch', 'sample_id', 'age', 'age_group', 'sex'
    layers: None (.X)
donors: 166
cell types: 9
batches: 14


## Select well-supported major cell types

In [5]:
MIN_CELLS = 5000
MIN_DONORS = 20
MAX_CELLS_PER_DONOR_CELLTYPE = 100
SEED = 0

celltype_summary = (
    adata.obs
    .groupby(
        "cell_type",
        observed=True,
    )
    .agg(
        n_cells=(
            "cell_type",
            "size",
        ),
        n_donors=(
            "donor_id",
            "nunique",
        ),
    )
    .sort_values(
        [
            "n_donors",
            "n_cells",
        ],
        ascending=False,
    )
)

keep_celltypes = (
    celltype_summary.index[
        (
            celltype_summary[
                "n_cells"
            ]
            >= MIN_CELLS
        )
        & (
            celltype_summary[
                "n_donors"
            ]
            >= MIN_DONORS
        )
    ]
    .tolist()
)

display(
    celltype_summary
)

print(
    "Keeping:",
    keep_celltypes,
)

EXPECTED_CELLTYPES = {
    "CD4+ T cells",
    "Myeloid cells",
    "TRAV1-2- CD8+ T cells",
    "NK cells",
    "B cells",
    "gd T cells",
    "MAIT cells",
}

assert (
    set(keep_celltypes)
    == EXPECTED_CELLTYPES
)

adata_filt = adata[
    adata.obs[
        "cell_type"
    ].isin(
        keep_celltypes
    )
].copy()

celltype_summary.to_csv(
    QC_DIR
    / "celltype_support_full.csv"
)

,n_cells,n_donors
cell_type,,
CD4+ T cells,901152,166
Myeloid cells,336935,166
TRAV1-2- CD8+ T cells,313343,166
NK cells,205469,166
B cells,71614,166
gd T cells,60325,166
MAIT cells,24245,166
DN T cells,1490,163
Progenitor cells,1794,160


Keeping: ['CD4+ T cells', 'Myeloid cells', 'TRAV1-2- CD8+ T cells', 'NK cells', 'B cells', 'gd T cells', 'MAIT cells']


## Fixed donor × cell-type subsampling

In [6]:
def subsample_by_group(
    adata,
    group_cols,
    max_cells_per_group=100,
    seed=0,
):
    # Exact legacy behavior.
    rng = np.random.default_rng(
        seed
    )

    keep = []

    groups = (
        adata.obs
        .groupby(
            group_cols,
            observed=True,
        )
        .indices
    )

    for _, idx in groups.items():
        idx = np.asarray(
            idx
        )

        if (
            len(idx)
            > max_cells_per_group
        ):
            idx = rng.choice(
                idx,
                size=max_cells_per_group,
                replace=False,
            )

        keep.extend(
            idx
        )

    keep = np.asarray(
        keep
    )

    rng.shuffle(
        keep
    )

    return adata[
        keep
    ].copy()

adata_bench = (
    subsample_by_group(
        adata_filt,
        group_cols=[
            "donor_id",
            "cell_type",
        ],
        max_cells_per_group=
            MAX_CELLS_PER_DONOR_CELLTYPE,
        seed=SEED,
    )
)

print(adata_bench)

print(
    adata_bench.obs[
        "cell_type"
    ].value_counts()
)

print(
    "donors:",
    adata_bench.obs[
        "donor_id"
    ].nunique(),
)

print(
    "batches:",
    adata_bench.obs[
        "batch"
    ].nunique(),
)

assert (
    adata_bench.n_obs
    == 108_682
)

assert (
    adata_bench.n_vars
    == 36_601
)

assert (
    adata_bench.obs[
        "donor_id"
    ].nunique()
    == 166
)

assert (
    adata_bench.obs[
        "cell_type"
    ].nunique()
    == 7
)

assert (
    adata_bench.obs[
        "batch"
    ].nunique()
    == 14
)

assert (
    "counts"
    not in adata_bench.layers
)

AnnData object with n_obs × n_vars = 108682 × 36601
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers', 'donor_id', 'cell_type', 'batch', 'sample_id', 'age', 'age_group', 'sex'
    layers: None (.X)
cell_type
Myeloid cells            16600
CD4+ T cells             16600
TRAV1-2- CD8+ T cells    16575
NK cells                 16464
B cells                  16297
gd T cells               14085
MAIT cells               12061
Name: count, dtype: int64
donors: 166
batches: 14


## Final Stage-0 provenance and QC

In [7]:
celltype_summary_final = (
    adata_bench.obs
    .groupby(
        "cell_type",
        observed=True,
    )
    .agg(
        n_cells=(
            "cell_type",
            "size",
        ),
        n_donors=(
            "donor_id",
            "nunique",
        ),
        n_batches=(
            "batch",
            "nunique",
        ),
    )
    .sort_values(
        "n_cells",
        ascending=False,
    )
)

display(
    celltype_summary_final
)

expected_counts = {
    "Myeloid cells": 16600,
    "CD4+ T cells": 16600,
    "TRAV1-2- CD8+ T cells": 16575,
    "NK cells": 16464,
    "B cells": 16297,
    "gd T cells": 14085,
    "MAIT cells": 12061,
}

observed_counts = (
    adata_bench.obs[
        "cell_type"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
)

assert (
    observed_counts
    == expected_counts
)

donor_summary = (
    adata_bench.obs
    .groupby(
        "donor_id",
        observed=True,
    )
    .agg(
        n_cells=(
            "donor_id",
            "size",
        ),
        n_cell_types=(
            "cell_type",
            "nunique",
        ),
        batch=(
            "batch",
            lambda x:
                ",".join(
                    sorted(
                        map(
                            str,
                            x.unique(),
                        )
                    )
                ),
        ),
        age=(
            "age",
            "first",
        ),
        age_group=(
            "age_group",
            "first",
        ),
        sex=(
            "sex",
            "first",
        ),
    )
    .sort_values(
        "n_cells",
        ascending=False,
    )
)

donor_batch = (
    adata_bench.obs[
        [
            "donor_id",
            "batch",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "donor_id"
    )["batch"]
    .nunique()
)

print(
    "Donors appearing in multiple batches:",
    int(
        (
            donor_batch
            > 1
        ).sum()
    ),
)

celltype_summary_final.to_csv(
    QC_DIR
    / "celltype_summary_final.csv"
)

donor_summary.to_csv(
    QC_DIR
    / "donor_summary.csv"
)

(
    adata_bench.obs[
        "batch"
    ]
    .value_counts()
    .to_csv(
        QC_DIR
        / "batch_counts.csv",
        header=[
            "n_cells",
        ],
    )
)

pd.Series(
    adata_bench.obs_names.astype(str),
    name="obs_name",
).to_csv(
    QC_DIR
    / "selected_cells.csv",
    index=False,
)

summary = {
    "source_adata_path":
        str(ADATA_PATH),
    "source_metadata_path":
        str(META_PATH),
    "expression_provenance":
        "contributor-provided normalized/transformed expression",
    "raw_gene_level_counts_available":
        False,
    "normalization_reapplied":
        False,
    "scvi_supported":
        False,
    "min_cells_per_celltype":
        MIN_CELLS,
    "min_donors_per_celltype":
        MIN_DONORS,
    "max_cells_per_donor_celltype":
        MAX_CELLS_PER_DONOR_CELLTYPE,
    "subsample_seed":
        SEED,
    "n_cells_final":
        int(adata_bench.n_obs),
    "n_genes_final":
        int(adata_bench.n_vars),
    "n_donors_final":
        int(
            adata_bench.obs[
                "donor_id"
            ].nunique()
        ),
    "n_cell_types_final":
        int(
            adata_bench.obs[
                "cell_type"
            ].nunique()
        ),
    "n_batches_final":
        int(
            adata_bench.obs[
                "batch"
            ].nunique()
        ),
    "output_h5ad":
        str(OUT_H5AD),
}

with (
    QC_DIR
    / "stage0_summary.json"
).open(
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

display(
    pd.DataFrame(
        [
            {
                "parameter": k,
                "value": v,
            }
            for k, v
            in summary.items()
        ]
    )
)

,n_cells,n_donors,n_batches
cell_type,,,
CD4+ T cells,16600,166,14
Myeloid cells,16600,166,14
TRAV1-2- CD8+ T cells,16575,166,14
NK cells,16464,166,14
B cells,16297,166,14
gd T cells,14085,166,14
MAIT cells,12061,166,14


Donors appearing in multiple batches: 99


,parameter,value
0,source_adata_path,/fastscratch/myscratch/xchen5/all_pbmcs/all_pb...
1,source_metadata_path,/fastscratch/myscratch/xchen5/all_pbmcs/all_pb...
2,expression_provenance,contributor-provided normalized/transformed ex...
3,raw_gene_level_counts_available,False
4,normalization_reapplied,False
5,scvi_supported,False
6,min_cells_per_celltype,5000
7,min_donors_per_celltype,20
8,max_cells_per_donor_celltype,100
9,subsample_seed,0


## Save the Stage-0 checkpoint

In [8]:
if OUT_H5AD.exists():
    raise FileExistsError(
        f"{OUT_H5AD} already exists. "
        "Rename/remove it before rerunning Stage 0."
    )

adata_bench.write_h5ad(
    OUT_H5AD,
    compression="gzip",
)

print(
    "Saved:",
    OUT_H5AD,
)

print(
    f"Size: "
    f"{OUT_H5AD.stat().st_size / 1e9:.2f} GB"
)

Saved: /users/xchen5/scRNA-cross-donor-generalization/data/blood_atlas/blood_atlas_subsampled.h5ad
Size: 0.35 GB


## Reload sanity check

In [9]:
check = ad.read_h5ad(
    OUT_H5AD,
    backed="r",
)

print(check)
print(
    "layers:",
    list(
        check.layers.keys()
    ),
)

assert (
    check.n_obs
    == 108_682
)

assert (
    check.n_vars
    == 36_601
)

assert (
    "counts"
    not in check.layers
)

if (
    hasattr(
        check,
        "file",
    )
    and check.file
    is not None
):
    check.file.close()

print(
    "Stage 0 complete. "
    "No raw counts were manufactured."
)

AnnData object with n_obs × n_vars = 108682 × 36601 backed at '/users/xchen5/scRNA-cross-donor-generalization/data/blood_atlas/blood_atlas_subsampled.h5ad'
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers', 'donor_id', 'cell_type', 'batch', 'sample_id', 'age', 'age_group', 'sex'
    layers: None (.X)
layers: [None]
Stage 0 complete. No raw counts were manufactured.
